# CrivoNews — Passo 4: fine-tuning do Llama 3.1 8B com LoRA (QLoRA + Unsloth)

Roda no **Google Colab** (Ambiente de execução → Alterar tipo → **GPU T4**) ou no **Kaggle**
(Settings → Accelerator → **GPU T4 x2** ou P100; Internet ligada).

Entrada: `treino.jsonl` e `validacao.jsonl` gerados por `3_montar_dataset.py`.
Saída: `crivo.Q4_K_M.gguf` (~4,9 GB), que vai para o Ollama como `meu-llama-tcc`.

Execute as células em ordem. Tempo estimado na T4: 1–2 h para ~3.000 exemplos e 2 épocas.

In [ ]:
%%capture
!pip install unsloth

## 1. Dados
- **Colab:** coloque `treino.jsonl` e `validacao.jsonl` na pasta **Meu Drive/crivonews** (a mesma deste notebook).
  A célula abaixo conecta o Drive (autorize na janela que abrir) e os encontra; se não achar, abre um botão de upload.
- **Kaggle:** crie um Dataset com os dois arquivos e adicione ao notebook (*Add Input*); a célula os encontra sozinha.

In [ ]:
import glob, os, shutil

PASTA_DRIVE = "/content/drive/MyDrive/crivonews"
try:
    from google.colab import drive  # só existe no Colab
    drive.mount("/content/drive")
except ImportError:
    pass  # Kaggle

def achar(nome):
    candidatos = (glob.glob(nome) + glob.glob(f"{PASTA_DRIVE}/{nome}")
                  + glob.glob(f"/kaggle/input/**/{nome}", recursive=True))
    return candidatos[0] if candidatos else None

if not (achar("treino.jsonl") and achar("validacao.jsonl")):
    from google.colab import files
    files.upload()

ARQ_TREINO, ARQ_VAL = achar("treino.jsonl"), achar("validacao.jsonl")
print(ARQ_TREINO, ARQ_VAL)

## 2. Modelo base (4 bits) + adaptadores LoRA

In [ ]:
from unsloth import FastLanguageModel

MAX_SEQ = 2048

model, tokenizer = FastLanguageModel.from_pretrained(
    model_name="unsloth/Meta-Llama-3.1-8B-Instruct-bnb-4bit",
    max_seq_length=MAX_SEQ,
    load_in_4bit=True,
)

model = FastLanguageModel.get_peft_model(
    model,
    r=16,
    lora_alpha=16,
    lora_dropout=0,
    target_modules=["q_proj", "k_proj", "v_proj", "o_proj", "gate_proj", "up_proj", "down_proj"],
    bias="none",
    use_gradient_checkpointing="unsloth",
    random_state=42,
)

## 3. Formatação no template do Llama 3

Usamos o template `llama-3` (sem o bloco de sistema com datas que o `llama-3.1` injeta) porque é
**exatamente** o que o `Modelfile` do Ollama monta na inferência: o backend manda tudo como
mensagem de usuário. O `<|begin_of_text|>` é removido do texto porque o tokenizer já o adiciona.

In [ ]:
from datasets import load_dataset
from unsloth.chat_templates import get_chat_template

tokenizer = get_chat_template(tokenizer, chat_template="llama-3")

def formatar(lote):
    textos = [tokenizer.apply_chat_template(m, tokenize=False, add_generation_prompt=False)
              for m in lote["messages"]]
    return {"text": [t.removeprefix(tokenizer.bos_token) for t in textos]}

dados = load_dataset("json", data_files={"train": ARQ_TREINO, "validation": ARQ_VAL})
dados = dados.map(formatar, batched=True)

# Descarta exemplos que não cabem no contexto (seriam truncados e perderiam o JSON de resposta)
antes = {k: len(v) for k, v in dados.items()}
dados = dados.filter(lambda ex: len(tokenizer(ex["text"]).input_ids) <= MAX_SEQ)
print("antes:", antes, "| depois:", {k: len(v) for k, v in dados.items()})
print(dados["train"][0]["text"][-700:])

## 4. Treino (o erro só é calculado sobre a resposta JSON, não sobre o prompt)

In [ ]:
from trl import SFTTrainer, SFTConfig
from unsloth.chat_templates import train_on_responses_only

trainer = SFTTrainer(
    model=model,
    tokenizer=tokenizer,
    train_dataset=dados["train"],
    eval_dataset=dados["validation"],
    args=SFTConfig(
        dataset_text_field="text",
        per_device_train_batch_size=2,
        gradient_accumulation_steps=4,   # lote efetivo = 8
        num_train_epochs=2,
        learning_rate=2e-4,
        warmup_ratio=0.03,
        lr_scheduler_type="linear",
        optim="adamw_8bit",
        weight_decay=0.01,
        logging_steps=10,
        eval_strategy="steps",
        eval_steps=100,
        save_strategy="steps",
        save_steps=200,
        save_total_limit=2,
        seed=42,
        output_dir="outputs",
        report_to="none",
    ),
)

trainer = train_on_responses_only(
    trainer,
    instruction_part="<|start_header_id|>user<|end_header_id|>\n\n",
    response_part="<|start_header_id|>assistant<|end_header_id|>\n\n",
)

estatisticas = trainer.train()
print(estatisticas)

Guarde o histórico de *loss* (treino e validação) — o gráfico vai bem no capítulo de resultados do TFG.

In [ ]:
import pandas as pd
historico = pd.DataFrame(trainer.state.log_history)
historico.to_csv("historico_treino.csv", index=False)
historico[["step", "loss", "eval_loss"]].groupby("step").first().plot(marker=".", title="Loss")

## 5. Teste rápido com um exemplo da validação

In [ ]:
import json
FastLanguageModel.for_inference(model)

exemplo = dados["validation"][0]["messages"]
entrada = tokenizer.apply_chat_template(exemplo[:1], add_generation_prompt=True, return_tensors="pt").to("cuda")
saida = model.generate(input_ids=entrada, max_new_tokens=300, temperature=0.2, do_sample=True)
resposta = tokenizer.decode(saida[0][entrada.shape[1]:], skip_special_tokens=True)

print("Gerado:  ", resposta)
print("Esperado:", exemplo[1]["content"])
print("JSON válido:", bool(json.loads(resposta[resposta.find("{"):resposta.rfind("}") + 1])))

## 6. Exportação
Salva o adaptador LoRA (pequeno, bom para guardar) e o modelo completo em GGUF `q4_k_m` para o Ollama.
A conversão para GGUF demora alguns minutos.

In [ ]:
model.save_pretrained("crivo-lora")
tokenizer.save_pretrained("crivo-lora")

model.save_pretrained_gguf("crivo-gguf", tokenizer, quantization_method="q4_k_m")

# O nome do arquivo gerado muda entre versões do Unsloth; padroniza para o que o Modelfile espera
gerados = [g for g in glob.glob("crivo-gguf*/**/*.gguf", recursive=True) + glob.glob("*.gguf")
           if "q4_k_m" in g.lower()]
print("GGUF gerados:", gerados)
shutil.move(gerados[0], "crivo.Q4_K_M.gguf")
print("Pronto:", round(os.path.getsize("crivo.Q4_K_M.gguf") / 1e9, 2), "GB")

## 7. Baixar o resultado
- **Kaggle:** clique em *Save Version → Save & Run All (Commit)* e baixe `crivo.Q4_K_M.gguf` na aba *Output*.
- **Colab:** a célula abaixo copia tudo para **Meu Drive/crivonews**; baixe de lá (direto pelo navegador, 5 GB costuma falhar).

Salve o arquivo **fora do OneDrive** (ex.: `C:\modelos\crivo\`) e siga o README a partir de "Importar no Ollama".

In [ ]:
if os.path.isdir("/content/drive/MyDrive"):
    os.makedirs(PASTA_DRIVE, exist_ok=True)
    for arq in ["crivo.Q4_K_M.gguf", "historico_treino.csv"]:
        shutil.copy(arq, PASTA_DRIVE)
    shutil.copytree("crivo-lora", f"{PASTA_DRIVE}/crivo-lora", dirs_exist_ok=True)
    print("Copiado para", PASTA_DRIVE)
else:
    print("Drive não conectado (Kaggle?): baixe os arquivos pela aba Output.")